<table border=1 width='99%'>
<tr>
<td bgcolor='bllue'>

# **<font color="#FFFFFF"> PC3 - CD1 - Desarrollo de un Modelo de Clasificación de Imágenes </font>**
</td>
</tr>
</table>

<table width='99%'>
<tr>
<td bgcolor='#FFBA39'>

## **<font color="#000000"> Introducción</font>**
</td>
</tr>
</table>

El proceso llevado a cabo comprende varias etapas fundamentales:

1. **Descarga de imágenes desde Google**, utilizando herramientas automatizadas para ampliar el dataset original.  
2. **Organización de las imágenes por clases**, siguiendo una estructura clara en carpetas que permita su uso en el entrenamiento del modelo.  
3. **Limpieza, selección y etiquetado de las imágenes**, asegurando que cada clase tenga imágenes válidas, relevantes y de buena calidad.  
4. **Entrenamiento del modelo CNN** utilizando TensorFlow/Keras.  
5. **Optimización de los hiperparámetros** con el fin de mejorar la precisión del modelo.  
6. **Implementación de una interfaz sencilla**, donde el usuario puede subir una imagen y el modelo devuelve la predicción correspondiente.


## 1. Descarga imágenes desde Google

Se realizó una descarga de imágenes utilizando la librería **google_images_download**. 

#### Clases trabajadas


Las clases seleccionadas para el dataset son:

- **daisy**  
- **dandelion**  
- **rose**  
- **sunflower**  
- **tulip**

Cada clase tiene su propia subcarpeta

In [4]:
!pip install google_images_download

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 9.7/9.7 MB 46.3 MB/s  0:00:00
  Created wheel for google_images_download: filename=google_images_download-2.8.0-py2.py3-none-any.whl size=14611 sha256=aaa297696a4613c2c0707a87d9238ff49a1562fcdbb19e4185382d06bb416d0d
  Stored in directory: c:\users\lenovo\appdata\local\pip\cache\wheels\dc\83\37\7303b15f3e8a5bfbd5c7ebbfe13f0c666ada6f8efecc6d77ec
Successfully built google_images_download

  Attempting uninstall: websocket-client

    Found existing installation: websocket-client 1.5.1

    Uninstalling websocket-client-1.5.1:

      Successfully uninstalled websocket-client-1.5.1

   --- ------------------------------------  1/11 [websocket-client]
   --- ------------------------------------  1/11 [websocket-client]
  Attempting uninstall: h11
   --- ---------------------------

  DEPRECATION: Building 'google_images_download' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'google_images_download'. Discussion can be found at https://github.com/pypa/pip/issues/6334
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpcore 0.9.1 requires h11<0.10,>=0.8, but you have h11 0.16.0 which is incompatible.
httpx 0.13.3 requires idna==2.*, but you have idna 3.10 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from google_images_download import google_images_download

# Carpeta train (CAMBIA ESTA RUTA)
train_path = r"C:/Users/Lenovo/Downloads/pc3CD1/archive (14)/train"  # ejemplo: r"C:/Users/Lucero/Downloads/pc3CD1/archive/train"

clases = ["daisy", "dandelion", "rose", "sunflower", "tulip"]

imagenes_por_clase = 50   # <-- cantidad que quieres agregar por clase

response = google_images_download.googleimagesdownload()

for clase in clases:
    argumentos = {
        "keywords": clase + " flower",
        "limit": imagenes_por_clase,
        "output_directory": train_path,
        "image_directory": clase,
        "print_urls": False,
        "safe_search": True,
        "format": "jpg"
    }

    print(f"Descargando imágenes para: {clase} ...")
    response.download(argumentos)

print("✔ Descarga completada.")


Descargando imágenes para: daisy ...

Item no.: 1 --> Item name = daisy flower
Evaluating...
Starting Download...


Unfortunately all 50 could not be downloaded because some images were not downloadable. 0 is all we got for this search filter!

Errors: 0

Descargando imágenes para: dandelion ...

Item no.: 1 --> Item name = dandelion flower
Evaluating...
Starting Download...


Unfortunately all 50 could not be downloaded because some images were not downloadable. 0 is all we got for this search filter!

Errors: 0

Descargando imágenes para: rose ...

Item no.: 1 --> Item name = rose flower
Evaluating...
Starting Download...


Unfortunately all 50 could not be downloaded because some images were not downloadable. 0 is all we got for this search filter!

Errors: 0

Descargando imágenes para: sunflower ...

Item no.: 1 --> Item name = sunflower flower
Evaluating...
Starting Download...


Unfortunately all 50 could not be downloaded because some images were not downloadable. 0 is all we go

Se descargaron un total **2.695 archivos imágenes nuevas**

| Clase       | Imágenes nuevas |
|-------------|-----------------|
| Daisy       | 501             |
| Dandelion   |  646            |
| Rose        | 468            |
| Sunflower   | 473              |
| Tulip       | 607          |

### Organizar en carpetas por clases

In [7]:
import os

train_path = r"C:/Users/Lenovo/Downloads/pc3CD1/train"
clases = ["daisy", "dandelion", "rose", "sunflower", "tulip"]

for c in clases:
    ruta = os.path.join(train_path, c)
    total = len(os.listdir(ruta))
    print(f"{c}: {total} imágenes")

daisy: 501 imágenes
dandelion: 646 imágenes
rose: 468 imágenes
sunflower: 473 imágenes
tulip: 607 imágenes


### Etiquetar y limpiar imágenes

In [8]:
from PIL import Image
import os

def es_valida(path):
    try:
        img = Image.open(path)
        img.verify()  
        img = Image.open(path)
        return img.size[0] > 100 and img.size[1] > 100
    except:
        return False

train_path = r"C:/Users/Lenovo/Downloads/pc3CD1/train"

clases = ["daisy", "dandelion", "rose", "sunflower", "tulip"]

for c in clases:
    carpeta = os.path.join(train_path, c)
    archivos = os.listdir(carpeta)
    
    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        if not es_valida(ruta):
            os.remove(ruta)
            print(f"Eliminada: {ruta}")


Después de la descarga masiva de imágenes (2,695 nuevas imágenes), se realizó un proceso de limpieza para asegurar la calidad del dataset.  
Las acciones realizadas fueron:

- Eliminación de imágenes dañadas o corruptas.
- Eliminación de imágenes muy pequeñas (<100 px).
- Eliminación de imágenes duplicadas.
- Eliminación manual de imágenes que no correspondían a la clase (objetos, personas, autos, etc.).
- Verificación final de la estructura de carpetas.

Finalmente, el dataset quedó listo para la etapa de entrenamiento del modelo CNN.